## DEFINING FUNCTIONS & IMPORTING LIBRARIES

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from joblib import Parallel, delayed

import camb
import healpy as hp

from sbi.utils import BoxUniform
from sbi.inference import NPE
from sbi.analysis import pairplot, plot_summary


/home/vasir/cmb-work/.venv/lib/python3.12/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [2]:

# Defining Constants
_ = torch.manual_seed(42)
_ = np.random.seed(0)

lmax = 3000
beam_fwhm = 5.0  # arcmin
BEAM = hp.gauss_beam(beam_fwhm / 60 / 180 * np.pi, lmax=lmax)  # length lmax+1

# White noise spectrum (choose whatever you used in Compression.ipynb, but make it length lmax+1)
nl_base = np.ones(lmax + 1) * (20/60/180*60)**2
nl_split = {
    "split1": nl_base,
    "split2": 1.3 * nl_base,
    "none":   np.zeros(lmax + 1),
}

# Fiducial CAMB params (same as Compression.ipynb)
param_dict = {
    "H0": 67.5, "ombh2": 0.022, "omch2": 0.122,
    "mnu": 0.06, "omk": 0.0, "tau": 0.06,
    "As": 2e-9, "ns": 0.965,
    "halofit_version": "mead",
    "lmax": lmax
}

# Parameters we infer/compress (note As is treated as As10 ≡ As*1e10 in the compressor)
derivatives = {"H0": 0.001, "ombh2": 0.005, "omch2": 0.005, "As": 0.005, "ns": 0.005}

# We'll keep a global compressor like your LV global summariser
COMPRESSOR = None


# -Compression
def getCompression(param_dict, derivatives, beam_fwhm=None, noise_cl=None):
    """
    Builds a MOPED/score-like compressor around a fiducial cosmology:
      t(d) = (∂μ/∂θ) C^{-1} (d-μ)
    where μ is the mean observed TT spectrum (including beam^2 and noise),
    and C is diagonal Gaussian variance approx.
    """
    lmax = param_dict["lmax"]
    ell = np.arange(lmax + 1)

    def getSpectrum(params):
        pars = camb.set_params(**params)
        results = camb.get_results(pars)
        cl_tt = results.get_cmb_power_spectra(pars, CMB_unit="muK", raw_cl=True)["total"][:lmax+1, 0]

        # beam^2 and noise in mean
        if beam_fwhm is not None and beam_fwhm != 0:
            beam = hp.gauss_beam(beam_fwhm / 60 / 180 * np.pi, lmax=lmax)
        else:
            beam = np.ones(lmax + 1)

        noise = np.zeros(lmax + 1) if noise_cl is None else noise_cl[:lmax+1]
        mu = cl_tt * beam**2 + noise
        return mu[2:]  # drop ell=0,1

    fiducial = getSpectrum(param_dict)  # length (lmax-1)

    # diagonal Gaussian variance: Var(Ĉ_ell) ≈ 2/(2ell+1) * C_ell^2   (full-sky approx)
    ell_use = np.arange(2, lmax + 1)
    cov_diag = 2.0 / (2 * ell_use + 1) * fiducial**2  # length (lmax-1)

    # derivatives wrt parameters (central differences)
    derivs = []
    for name, frac in derivatives.items():
        step = param_dict[name] * frac

        p_up = dict(param_dict);   p_up[name] = p_up[name] + step
        p_dn = dict(param_dict);   p_dn[name] = p_dn[name] - step

        up = getSpectrum(p_up)
        dn = getSpectrum(p_dn)

        # As is reported in "As10 = As*1e10" units for conditioning
        denom = 2 * step
        if name == "As":
            denom *= 1e10

        derivs.append((up - dn) / denom)

    derivs = np.asarray(derivs)  # shape (5, lmax-1)

    def compressor(cl_hat):
        """
        cl_hat is an estimated TT spectrum (length lmax+1).
        Returns a 5-vector summary.
        """
        d = cl_hat[:lmax+1][2:]  # drop ell=0,1
        return derivs @ ((d - fiducial) / cov_diag)

    return fiducial, cov_diag, derivs, compressor


def set_compressor_for_split(split_name):
    """
    Call this before generate_x(...) so summarize_simulation uses the right compressor
    (because the mean spectrum includes the split noise level).
    """
    global COMPRESSOR
    _, _, _, compressor = getCompression(
        param_dict,
        derivatives,
        beam_fwhm=beam_fwhm,
        noise_cl=nl_split[split_name]
    )
    COMPRESSOR = compressor


# Defining simulators
def theta_to_camb_params(theta_vec):
    """
    theta_vec order: [H0, ombh2, omch2, As10, ns]
    where As10 = As * 1e10.
    """
    H0, ombh2, omch2, As10, ns = theta_vec
    p = dict(param_dict)
    p["H0"] = float(H0)
    p["ombh2"] = float(ombh2)
    p["omch2"] = float(omch2)
    p["As"] = float(As10) * 1e-10
    p["ns"] = float(ns)
    return p


def simulate_total(parameters, t_span=None):
    """
    LV analogue: generate a 'latent truth' observation.
    Here: draw one clean CMB sky alm from the theory spectrum at theta.
    """
    p = theta_to_camb_params(parameters)
    pars = camb.set_params(**p)
    results = camb.get_results(pars)
    cl_tt = results.get_cmb_power_spectra(pars, CMB_unit="muK", raw_cl=True)["total"][:lmax+1, 0]
    cmb_alm = hp.synalm(cl_tt, lmax=lmax)
    return cmb_alm


def add_noise_and_plot(simulation_result, distn, sigma_hare=None, sigma_lynx=None, time=None, do_plot=True):
    """
    LV analogue: take latent and add split-specific noise.
    Input: simulation_result = clean cmb_alm
    Output: cl_hat (estimated TT spectrum)
    """
    cmb_alm = simulation_result

    # apply beam
    alm_obs = hp.almxfl(cmb_alm, BEAM)

    # add split noise in harmonic space
    nl = nl_split[distn]
    noise_alm = hp.synalm(nl, lmax=lmax)
    alm_obs = alm_obs + noise_alm

    # estimate spectrum
    cl_hat = hp.alm2cl(alm_obs)

    if do_plot:
        ell = np.arange(lmax + 1)
        plt.figure(figsize=(7, 3))
        plt.loglog(ell[2:], (ell[2:]**2) * cl_hat[2:], label=f"{distn} (auto)")
        plt.xlabel(r"$\ell$")
        plt.ylabel(r"$\ell^2 \hat C_\ell^{TT}$")
        plt.legend()
        plt.tight_layout()

    return cl_hat


def summarize_simulation(simulation_result):
    """
    LV analogue: summary statistics.
    Here: apply the global compressor to a cl_hat.
    """
    if COMPRESSOR is None:
        raise RuntimeError("COMPRESSOR is None. Call set_compressor_for_split('split1'/'split2') first.")
    cl_hat = simulation_result
    return np.asarray(COMPRESSOR(cl_hat))


def simulator_distribution(distn, parameters, n_obs=None, sigma_hare=None, sigma_lynx=None, observation=None, t_span=None):
    """
    LV analogue: choose noise model and return a noisy dataset.
    Here: given theta, generate a new sky realisation + beam + split-noise -> cl_hat
    """
    p = theta_to_camb_params(parameters)
    pars = camb.set_params(**p)
    results = camb.get_results(pars)
    cl_tt = results.get_cmb_power_spectra(pars, CMB_unit="muK", raw_cl=True)["total"][:lmax+1, 0]

    cmb_alm = hp.synalm(cl_tt, lmax=lmax)
    alm_obs = hp.almxfl(cmb_alm, BEAM)

    nl = nl_split[distn]
    noise_alm = hp.synalm(nl, lmax=lmax)
    alm_obs = alm_obs + noise_alm

    cl_hat = hp.alm2cl(alm_obs)
    return cl_hat


# -Defining Priors, stick to uniform
def define_uniform_prior():
    """
    Prior over [H0, ombh2, omch2, As10, ns]
    (pick ranges that reasonably cover your intended cosmology region)
    """
    low  = torch.tensor([60.0, 0.020, 0.105, 10.0, 0.92])
    high = torch.tensor([75.0, 0.024, 0.140, 35.0, 1.02])
    return BoxUniform(low=low, high=high)


def choose_prior_and_generate_theta(distn):
    if distn != "uniform":
        raise ValueError("For now use distn='uniform' for CMB.")
    prior = define_uniform_prior()
    theta = prior.sample((10_000,))  # same as your LV notebook
    return prior, theta


# generating data
# get rid of junk parameters
def parallel_simulate(theta, distn, n_obs=None, sigma_hare=None, sigma_lynx=None, observation=None, t_span=None):
    theta_np = theta.numpy()
    num_workers = 4  # CAMB is heavy; start smaller than your LV=8
    outs = Parallel(n_jobs=num_workers)(
        delayed(simulator_distribution)(distn, batch, n_obs, sigma_hare, sigma_lynx, observation, t_span)
        for batch in theta_np
    )
    return np.asarray(outs)


def generate_x(theta, distn, n_obs=None, sigma_hare=None, sigma_lynx=None, observation=None, t_span=None):
    simulation_outputs = parallel_simulate(theta, distn, n_obs, sigma_hare, sigma_lynx, observation, t_span)
    x = torch.as_tensor(np.asarray([summarize_simulation(sim) for sim in simulation_outputs]), dtype=torch.float32)
    return x


def plot_checker(x, x_obs):
    _ = pairplot(
        samples=x,
        points=x_obs[None, :],
        figsize=(4, 4),
    )



In [3]:

# trainig models
def train_net_generate_samples(x, theta, x_obs, prior, verbose, max_epoch, true_val):
    inference = NPE(prior=prior, density_estimator="nsf")
    posterior_net = inference.append_simulations(theta, x).train(max_num_epochs=max_epoch, stop_after_epochs=30)
    posterior_direct = inference.build_posterior(density_estimator=posterior_net, sample_with="direct")
    posterior_mcmc = inference.build_posterior(density_estimator=posterior_net, sample_with="mcmc")

    samples = posterior_mcmc.sample((1_000,), x=x_obs)

    if verbose:
        _ = plot_summary(inference, tags=["training_loss", "validation_loss"], figsize=(10, 2))
        print(posterior_mcmc)
        print("Observation x_obs:", x_obs)

        _ = pairplot(
            samples,
            points=true_val[None, :],
            figsize=(5, 5),
            labels=[r"$H_0$", r"$\Omega_b h^2$", r"$\Omega_c h^2$", r"$A_s\times 10^{10}$", r"$n_s$"]
        )

    return samples, posterior_mcmc, posterior_direct, inference


## TRAINING MODELS

In [4]:
# True parameters (order: [H0, ombh2, omch2, As10, ns])
true_parameters = np.asarray([67.5, 0.022, 0.122, 2e-9 * 1e10, 0.965])

# Latent "truth" sky (clean alm) — like your LV simulate_total(...)
observation = simulate_total(true_parameters)

prior_name = "uniform"
max_epoch = 1000


In [5]:

# split 1
set_compressor_for_split("split1")  # sets global COMPRESSOR to match this split noise

prior1, theta1 = choose_prior_and_generate_theta(prior_name)
x1 = generate_x(theta1, "split1", observation=observation)

clhat_obs1 = add_noise_and_plot(observation, "split1", do_plot=True)
x_obs1 = torch.as_tensor(summarize_simulation(clhat_obs1), dtype=torch.float32)

plot_checker(x1, x_obs1.numpy())


KeyboardInterrupt: 

In [ ]:

# split 2
set_compressor_for_split("split2")

prior2, theta2 = choose_prior_and_generate_theta(prior_name)
x2 = generate_x(theta2, "split2", observation=observation)

clhat_obs2 = add_noise_and_plot(observation, "split2", do_plot=True)
x_obs2 = torch.as_tensor(summarize_simulation(clhat_obs2), dtype=torch.float32)

plot_checker(x2, x_obs2.numpy())


In [ ]:

samples1, posterior1_mcmc, posterior1_direct, inference1 = train_net_generate_samples(
    x1, theta1, x_obs1, prior1, verbose=True, max_epoch=max_epoch, true_val=true_parameters
)


In [ ]:
samples2, posterior2_mcmc, posterior2_direct, inference2 = train_net_generate_samples(
    x2, theta2, x_obs2, prior2, verbose=True, max_epoch=max_epoch, true_val=true_parameters
)
